<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/generation-3_Florence-2-large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVLM Dashboard Insight Generation

## Initial Steps

In [1]:
!pip install -q supabase pillow requests torch torchvision einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.4 MB/s eta 0:00:00


In [2]:
import os
import time
import requests
import torch
from io import BytesIO
from PIL import Image
from supabase import create_client, Client
from google.colab import userdata
from huggingface_hub import login

In [3]:
# Supabase credentials
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client initialised.")

login(token=HF_TOKEN)
print("HuggingFace login successful.")

Supabase client initialised.
HuggingFace login successful.


In [4]:
# Pull qualifying metadata_ids from human_insights
hi_response = supabase.table("human_insights") \
    .select("metadata_id") \
    .eq("expected_dataset", True) \
    .is_("rejection_reason", "null") \
    .execute()

qualified_ids = list({row["metadata_id"] for row in hi_response.data})
print(f"Qualified dashboards: {len(qualified_ids)}")

# Pull metadata only for those ids
response = supabase.table("metadata") \
    .select("id, bucket_path") \
    .in_("id", qualified_ids) \
    .execute()
dashboards = response.data

print(f"Loaded {len(dashboards)} dashboards.")
print("Sample record:", dashboards[0] if dashboards else "(empty)")

Qualified dashboards: 40
Loaded 40 dashboards.
Sample record: {'id': 'abee2e83-6384-4c23-abfd-e5ede8b5a7bf', 'bucket_path': 'screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png'}


In [5]:
# Build public image URL from bucket_path
def build_image_url(bucket_path: str) -> str:
    return f"{SUPABASE_URL}/storage/v1/object/public/superstore/{bucket_path}"


# Fetch image from URL and return a PIL Image
def fetch_image(url: str) -> Image.Image:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")


# Extract model identity from a loaded model object
def get_model_meta(model, hf_id=None):
    cfg   = getattr(model, "config", None)
    hf_id = hf_id or getattr(cfg, "_name_or_path", None)
    name  = hf_id.split("/")[-1] if hf_id else None
    return {"model_name": name, "model_hf_id": hf_id}

In [6]:
# Prompt
PROMPT = """
You are a senior BI analyst generating structured insights for a business intelligence dashboard.
Think step-by-step: first extract visible quantitative facts, then identify visual patterns, then derive business implications.

Before writing any chart analysis, count the number of distinct charts visible in the dashboard and write: 'Chart count: N'.
Then produce exactly N chart analyses and no more.
Analyze chart-by-chart in Z-pattern (left to right, top to bottom).
If there are scoreboard/scorecard charts (e.g., sales, profit, orders, customers, etc), treat them as the first chart as one single chart with the title of "Scoreboard Overview".
Once grouped into Scoreboard Overview, those KPI panels are fully analyzed and must never appear again as individual charts anywhere in your output.
If there are no scoreboard/scorecard charts, proceed with writing the first chart available.
Do not treat UI labels, navigation tabs, filters, or sidebar controls as charts.
For each chart, write exactly:
L2: one sentence reporting only values explicitly shown or labeled: highest/lowest value, comparison, ranking, or proportion only. Do not compute anything not displayed in the image.
L3: one sentence describing a visual pattern: a direction, a shape, a gap, or an exception. Use natural language: "volatile", "dipped", "wider margin", "considerably far", "spread". Use hedging: "appears to", "seems to", "suggesting". Write NOT APPLICABLE if the chart is: a ranked table, a top-N list, or a gauge.
L4: one sentence connecting the pattern to business context or domain knowledge not visible in the chart. Must reference a specific value from L2 or a specific pattern from L3; never use generic phrases such as 'this could be due to' without grounding them in what was observed. Never restate what is already visible. Always required.

Output format:
Chart 1: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Chart 2: [Title]
L2: [One sentence.]
L3: [One sentence.] or NOT APPLICABLE
L4: [One sentence.]

Rules:
- Write EXACTLY 1 sentence per level per chart
- Skip navigation tabs, filters, sidebar controls, and dropdowns entierly
- Do not include axis labels, colors, or chart type names
- Immediately after writing your final chart analysis, write END OF ANALYSIS on its own line and generate no further text under any circumstances
"""

In [7]:
# Quick sanity check on the first dashboard
if dashboards:
    sample_url = build_image_url(dashboards[0]["bucket_path"])
    print("Sample URL:", sample_url)
    sample_img = fetch_image(sample_url)
    print("Image size:", sample_img.size)
    sample_img

Sample URL: https://olduvnqhykovcfbfouhe.supabase.co/storage/v1/object/public/superstore/screenshots/abee2e83-6384-4c23-abfd-e5ede8b5a7bf.png
Image size: (1200, 927)


---

## Model 3: Florence-2-large
**Source:** https://huggingface.co/microsoft/Florence-2-large  
**Parameters:** 0.8B  
**Architecture:**   
**Citation:** Xiao, B., Wu, H., Xu, W., Dai, X., Hu, H., Lu, Y., Zeng, M., Liu, C., & Yuan, L. (2024). Florence-2: Advancing a Unified Representation for a Variety of Vision Tasks. 2024 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR), 4818–4829. https://doi.org/10.1109/CVPR52733.2024.00461

In [18]:
!pip install -q transformers==4.44.2 tokenizers==0.19.1 --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 42.3 MB/s eta 0:00:00


In [17]:
!pip show tokenizers transformers | grep -E "Name|Version"

Name: tokenizers
Version: 0.19.1
Name: transformers
Version: 4.46.3


In [19]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
print("transformers:", transformers.__version__)

transformers: 4.46.3


In [21]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_3_HF_ID = "microsoft/Florence-2-large"

device_3    = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype_3 = torch.float16 if torch.cuda.is_available() else torch.float32

processor_3 = AutoProcessor.from_pretrained(
    MODEL_3_HF_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)
model_3 = AutoModelForCausalLM.from_pretrained(
    MODEL_3_HF_ID,
    torch_dtype=torch_dtype_3,
    trust_remote_code=True,
    token=HF_TOKEN,
).eval().to(device_3)

meta_3 = get_model_meta(model_3, hf_id=MODEL_3_HF_ID)
print(f"Loaded on {device_3}:")
print(f"  model_name:  {meta_3['model_name']}")
print(f"  model_hf_id: {meta_3['model_hf_id']}")

You are using a model of type florence2 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


AttributeError: TokenizersBackend has no attribute additional_special_tokens

In [ ]:
test_dashboard = dashboards[0]
test_image     = fetch_image(build_image_url(test_dashboard["bucket_path"]))

inputs = processor_3(
    text=PROMPT,
    images=test_image,
    return_tensors="pt"
).to(device_3, torch_dtype_3)

t0 = time.perf_counter()
generated_ids = model_3.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=1024,
    do_sample=False,
)
test_output = processor_3.batch_decode(generated_ids, skip_special_tokens=True)[0]
test_ms = int((time.perf_counter() - t0) * 1000)

print(f"Dashboard ID:   {test_dashboard['id']}")
print(f"Inference time: {test_ms} ms")
print(f"\nOutput:\n{test_output}")
display(test_image)

supabase.table("vlm_outputs").upsert({
    "metadata_id":       test_dashboard["id"],
    **meta_3,
    "raw_output":        test_output,
    "inference_success": True,
    "error_message":     None,
    "inference_ms":      test_ms,
}, on_conflict="metadata_id,model_name").execute()

print("Saved to vlm_outputs.")